## 1. Problem and learning goals

A local retail team supplies a CSV containing a **product name**, **quantity** and **unit price**. Your task is to create a small example dataset, read it without losing the column structure, and check that the import succeeded.

**Tasks from the slides**
1. Create a valid CSV file containing at least three products.
2. Read the CSV using `csv.DictReader` or pandas.
3. Display the rows and column names.
4. Explain and demonstrate how to handle a missing file.

**Success criteria:** A reproducible script writes, reads and validates three records. This notebook uses only Python's standard library; **pandas is not required**.

## 2. Prepare the imports and output location

The `csv` module handles CSV quoting correctly. `pathlib.Path` provides a clear, cross-platform way to work with paths.

To avoid overwriting a file that might already be called `products.csv`, we create a dedicated folder. The CSV **filename and sample values match the slide solution**.

In [1]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

data_dir = Path("lesson4_activity1_data")
data_dir.mkdir(parents=True, exist_ok=True)
path = data_dir / "products.csv"
print("CSV will be stored at:", path.resolve())

CSV will be stored at: /workspaces/ProgrammongForDA/app/L4/lesson4_activity1_data/products.csv


## 3. Task 1 — Write the CSV

We begin with a header row followed by three product records. Using `newline=""` is the recommended pattern for CSV files; it avoids extra blank lines on some platforms. `encoding="utf-8"` makes the text encoding explicit.

**Important:** Mode `"w"` creates a new file or replaces an existing file at this path. We use a dedicated exercise folder for safety.

In [2]:
rows = [
    ["product", "qty", "price"],
    ["Notebook", 3, 4.50],
    ["Pen", 10, 1.20],
    ["Folder", 2, 3.00],
]

with path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("File created:", path.name)
print("Exists:", path.is_file())

File created: products.csv
Exists: True


## 4. Task 2 — Read the CSV with `DictReader`

`csv.DictReader` uses the first row as **column names** and returns each remaining row as a dictionary. For example, one record will look like:

```python
{"product": "Notebook", "qty": "3", "price": "4.5"}
```

Notice that **CSV values are read as strings**, even when they look numeric. We convert them explicitly later.

In [3]:
with path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    column_names = reader.fieldnames
    data = list(reader)

print("Columns:", column_names)
print("Number of product records:", len(data))
print("First record:", data[0])

Columns: ['product', 'qty', 'price']
Number of product records: 3
First record: {'product': 'Notebook', 'qty': '3', 'price': '4.5'}


## 5. Task 3 — Display the rows and column names

Printing dictionaries preserves the relationship between values and their column names. This is safer than assuming the second value is always quantity or the third is always price.

In [4]:
print("COLUMN NAMES:", ", ".join(column_names))
print("\nPRODUCT ROWS:")
for number, record in enumerate(data, start=1):
    print(f"{number}. {record}")

COLUMN NAMES: product, qty, price

PRODUCT ROWS:
1. {'product': 'Notebook', 'qty': '3', 'price': '4.5'}
2. {'product': 'Pen', 'qty': '10', 'price': '1.2'}
3. {'product': 'Folder', 'qty': '2', 'price': '3.0'}


## 6. Validate the imported data

A file being readable does **not** automatically mean it contains the expected data. Check the column names, row count, missing values and product labels. `assert` is useful in teaching examples because it stops execution when an assumption is false.

In [5]:
expected_columns = ["product", "qty", "price"]
assert column_names == expected_columns, "Unexpected CSV column names"
assert len(data) == 3, "Expected exactly three products"
assert all(record["product"].strip() for record in data), "Blank product name"
assert all(record["qty"] != "" for record in data), "Missing quantity"
assert all(record["price"] != "" for record in data), "Missing price"

print("Validation passed: correct columns, three rows and no blank required fields.")

Validation passed: correct columns, three rows and no blank required fields.


## 7. Convert numeric fields before calculating

`"3" * "4.5"` is not a valid revenue calculation. Convert quantity to `int` and price to `float` before multiplying. Here we create a **new list** of typed records so the original imported strings remain available for comparison.

Revenue is calculated as:

\[\text{Revenue} = \text{Quantity} \times \text{Unit Price}\]

In [6]:
typed_products = []
for record in data:
    quantity = int(record["qty"])
    price = float(record["price"])
    assert quantity >= 0, "Quantity must not be negative"
    assert price >= 0, "Price must not be negative"
    typed_products.append({
        "product": record["product"],
        "qty": quantity,
        "price": price,
        "revenue": quantity * price,
    })

for item in typed_products:
    print(f"{item['product']:<10} {item['qty']:>2} × £{item['price']:>4.2f} = £{item['revenue']:>5.2f}")

total_revenue = sum(item["revenue"] for item in typed_products)
print(f"Total revenue: £{total_revenue:.2f}")
assert abs(total_revenue - 31.50) < 1e-9

Notebook    3 × £4.50 = £13.50
Pen        10 × £1.20 = £12.00
Folder      2 × £3.00 = £ 6.00
Total revenue: £31.50


## 8. Task 4 — Handle a missing file

If the input file is missing, Python raises `FileNotFoundError`. A helpful application can catch that error and explain what the user should check.

The function below first checks whether the path exists. The example uses a temporary directory so the demonstration is **guaranteed to use a missing path** without touching a real file.

In [7]:
def load_product_csv(csv_path):
    """Read a product CSV and return (column_names, rows).

    Parameters
    ----------
    csv_path : str or pathlib.Path
        Path to a CSV file with product, qty and price columns.

    Raises
    ------
    FileNotFoundError
        If the input file does not exist.
    ValueError
        If the required column names are missing.
    """
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    with csv_path.open("r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        columns = reader.fieldnames or []
        required = {"product", "qty", "price"}
        if not required.issubset(columns):
            raise ValueError("CSV must contain product, qty and price columns")
        return columns, list(reader)

verified_columns, verified_rows = load_product_csv(path)
assert verified_columns == expected_columns
assert len(verified_rows) == 3
print("Existing file loaded successfully.")

Existing file loaded successfully.


In [8]:
with TemporaryDirectory() as temp_folder:
    missing_path = Path(temp_folder) / "missing_products.csv"
    try:
        load_product_csv(missing_path)
    except FileNotFoundError as error:
        print("Missing file handled safely:", error.__class__.__name__)
        print("Suggested action: check the folder and filename, then retry.")

Missing file handled safely: FileNotFoundError
Suggested action: check the folder and filename, then retry.


## 9. Expected results and interpretation

| Check | Expected result |
|---|---|
| Columns | `product`, `qty`, `price` |
| Number of rows | `3` |
| Notebook revenue | `3 × £4.50 = £13.50` |
| Pen revenue | `10 × £1.20 = £12.00` |
| Folder revenue | `2 × £3.00 = £6.00` |
| **Total revenue** | **£31.50** |

**Why `DictReader`?** It preserves field names, which helps avoid accidentally interpreting a price as a quantity.

**Common mistakes:** Using `"w"` on an important existing file; using `split(",")` on CSV fields that may contain commas; forgetting `newline=""`; and multiplying string values without conversion.

## 10. Review questions (with model answers)

**Q1. Why use `with path.open(...)`?** It closes the file automatically, including if an exception occurs.

**Q2. What is the difference between `"w"` and `"a"`?** `"w"` overwrites an existing file; `"a"` appends new content to the end.

**Q3. Why not use `split(",")` for a real CSV?** A quoted field can contain a comma (e.g., `"Jones, Alice"`), and a CSV parser handles it correctly.

**Q4. How can we verify the import?** Check column names, record count, required values and numerical conversions.

**Optional extension:** Add a fourth product in a *separate* CSV and recalculate total revenue. How would you detect a negative price or a missing quantity?